# 02a - Global Re-Identification

هذا النوتبوك يربط هويات الأشخاص عبر الكاميرات المختلفة. يأخذ `local_tracks.csv` من Notebook 01 ويستخدم OSNet embeddings + Hungarian assignment + Homography لإنتاج `global_track_id` موحّد لكل شخص بغض النظر عن عدد الكاميرات التي ظهر فيها.

**المخرجات:**
> **Legacy only:** هذا النوتبوك لا يغذي الـDashboard ولا يكتب فوق مخرجات Notebook 02.

- `global_tracks_legacy.csv` — نسخة بحثية منفصلة
- `reid_mapping_legacy.csv` — mapping بحثي منفصل
- `reid_report_legacy.json` — تقرير بحثي منفصل

**المتطلبات:** Notebook 01 مكتمل + `camera_calibration.generated.json` + أوزان OSNet.

In [ ]:
from __future__ import annotations

from datetime import datetime, timezone
from pathlib import Path
import json
import sys

import numpy as np
import pandas as pd

# ──────────────────────────────────── paths ────────────────────────────────────
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'Notebook':
    PROJECT_ROOT = PROJECT_ROOT.parent

# Ensure the Notebook directory is importable for mtmc_reid
NOTEBOOK_DIR = PROJECT_ROOT / 'Notebook'
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

TABLES_DIR = PROJECT_ROOT / 'Output' / 'tables'
RAW_DIR = PROJECT_ROOT / 'Data' / 'raw'
CONFIG_DIR = PROJECT_ROOT / 'Data' / 'config'
CACHE_DIR = PROJECT_ROOT / 'Output' / 'cache'
CACHE_DIR.mkdir(parents=True, exist_ok=True)

LOCAL_TRACKS_PATH = TABLES_DIR / 'local_tracks.csv'
CALIBRATION_PATH = CONFIG_DIR / 'camera_calibration.generated.json'
REID_CONFIG_PATH = CONFIG_DIR / 'mtmc_reid_config.json'

if not LOCAL_TRACKS_PATH.exists():
    raise FileNotFoundError('local_tracks.csv is missing. Run Notebook 01 first.')

local_tracks = pd.read_csv(LOCAL_TRACKS_PATH)
required_columns = {
    'camera_id', 'frame_index', 'timestamp_sec', 'local_track_id',
    'confidence', 'x1', 'y1', 'x2', 'y2', 'foot_x', 'foot_y'
}
missing_columns = sorted(required_columns.difference(local_tracks.columns))
if missing_columns:
    raise ValueError(f'local_tracks.csv is missing required columns: {missing_columns}')
if local_tracks.empty:
    raise ValueError('local_tracks.csv is empty.')

n_cameras = local_tracks.camera_id.nunique()
print(f'Loaded {len(local_tracks):,} local detections from {n_cameras} camera(s).')

if n_cameras < 2:
    print('\n⚠ Only one camera found — ReID requires at least 2 cameras.')
    print('Saving a pass-through global_tracks_legacy.csv with unique global IDs per local track.')
    import re
    def _infer_store(cid):
        m = re.search(r'(place_\\d+)', str(cid), flags=re.IGNORECASE)
        return m.group(1).lower() if m else 'default_store'
    single_cam = local_tracks.copy()
    single_cam['store_id'] = single_cam['camera_id'].map(_infer_store)
    single_cam['global_track_id'] = [
        f'global_{i:06d}' for i in single_cam['local_track_id']
    ]
    single_cam.to_csv(TABLES_DIR / 'global_tracks_legacy.csv', index=False)
    mapping_df = single_cam[['store_id', 'camera_id', 'local_track_id', 'global_track_id']].drop_duplicates()
    mapping_df.to_csv(TABLES_DIR / 'reid_mapping_legacy.csv', index=False)
    report = {
        'schema_version': 1,
        'created_at_utc': datetime.now(timezone.utc).isoformat(),
        'cameras': 1,
        'status': 'single_camera_passthrough',
        'total_tracklets': int(single_cam.local_track_id.nunique()),
        'global_identities': int(single_cam.global_track_id.nunique()),
        'accepted_matches': 0,
        'rejected_matches': 0,
    }
    (TABLES_DIR / 'reid_report_legacy.json').write_text(json.dumps(report, indent=2), encoding='utf-8')
    print(f'Saved {len(single_cam):,} rows to global_tracks_legacy.csv (pass-through, no merging).')
    # Stop further cells from executing (they need 2+ cameras)
    SINGLE_CAMERA_MODE = True
else:
    SINGLE_CAMERA_MODE = False

In [ ]:
if SINGLE_CAMERA_MODE:
    print('Single-camera mode — skipping ReID pipeline.')
else:
    import mtmc_reid

    # ──────────────── Load configuration ────────────────
    reid_config = json.loads(REID_CONFIG_PATH.read_text(encoding='utf-8'))
    calibrations = json.loads(CALIBRATION_PATH.read_text(encoding='utf-8'))

    # ──────────────── Apply Homography ────────────────
    # Map foot points to a shared floor coordinate system
    print('Applying homographies to project foot points onto shared floor...')
    detections_with_floor = mtmc_reid.apply_homographies(local_tracks, calibrations)
    print(f'  Floor projection done. Calibration modes: '
          f'{detections_with_floor["calibration_mode"].value_counts().to_dict()}')

    # ──────────────── Summarize Tracklets ────────────────
    print('\nSummarizing tracklets...')
    tracklets = mtmc_reid.summarize_tracklets(detections_with_floor)
    print(f'  {len(tracklets):,} tracklets across {tracklets.camera_id.nunique()} cameras.')

In [ ]:
if SINGLE_CAMERA_MODE:
    print('Single-camera mode — skipping embedding extraction.')
else:
    # ──────────────── Extract OSNet Embeddings ────────────────
    embedding_config = reid_config['reid']
    embedding_cache_path = CACHE_DIR / 'tracklet_embeddings.npz'
    reuse_cache = reid_config.get('execution', {}).get('reuse_embedding_cache', True)

    if reuse_cache and embedding_cache_path.exists():
        print('Loading cached embeddings...')
        cached = np.load(embedding_cache_path, allow_pickle=True)
        embeddings = {str(key): cached[key] for key in cached.files}
        # Verify all current tracklets have cached embeddings
        current_ids = set(tracklets['tracklet_id'].astype(str))
        cached_ids = set(embeddings.keys())
        if not current_ids.issubset(cached_ids):
            print(f'  Cache is stale ({len(current_ids - cached_ids)} missing tracklets). Re-extracting...')
            reuse_cache = False
        else:
            embedding_report = {
                'requested_crops': 0,
                'valid_crops': 0,
                'embedded_tracklets': len(embeddings),
                'source': 'cache',
            }
            print(f'  Loaded {len(embeddings):,} cached embeddings.')

    if not reuse_cache or not embedding_cache_path.exists():
        print('Loading OSNet model...')
        model, device = mtmc_reid.load_osnet_model(
            PROJECT_ROOT,
            embedding_config['weights_path'],
            embedding_config.get('device', 'auto'),
        )
        print(f'  OSNet loaded on {device}.')

        print('Extracting embeddings from video crops...')
        embeddings, embedding_report = mtmc_reid.extract_tracklet_embeddings(
            detections_with_floor,
            raw_dir=RAW_DIR,
            model=model,
            device=device,
            samples_per_tracklet=embedding_config.get('samples_per_tracklet', 12),
            min_confidence=embedding_config.get('min_confidence', 0.25),
            min_box_area=embedding_config.get('min_box_area', 1024.0),
            batch_size=embedding_config.get('batch_size', 32),
        )
        # Cache for future runs
        np.savez_compressed(embedding_cache_path, **embeddings)
        print(f'  Extracted {embedding_report["embedded_tracklets"]:,} embeddings '
              f'from {embedding_report["valid_crops"]:,} crops. Cached to {embedding_cache_path.name}.')

In [ ]:
if SINGLE_CAMERA_MODE:
    print('Single-camera mode — skipping cross-camera association.')
else:
    # ──────────────── Attach Embeddings & Run Association ────────────────
    print('Attaching embeddings to tracklets...')
    tracklets_with_emb = mtmc_reid.attach_tracklet_embeddings(tracklets, embeddings)

    # Build association config from mtmc_reid_config.json
    assoc_params = reid_config.get('association', {})
    assoc_config = mtmc_reid.AssociationConfig(
        **{key: value for key, value in assoc_params.items()
           if key in mtmc_reid.AssociationConfig.__dataclass_fields__}
    )

    print('\nRunning cross-camera association pipeline...')
    print(f'  Config: appearance_weight={assoc_config.appearance_weight}, '
          f'spatial_weight={assoc_config.spatial_weight}, '
          f'temporal_weight={assoc_config.temporal_weight}')
    print(f'  Gates: max_appearance_dist={assoc_config.max_appearance_distance}, '
          f'max_floor_dist={assoc_config.max_floor_distance}, '
          f'max_time_gap={assoc_config.max_time_gap_sec}s')

    result = mtmc_reid.run_mtmc_association(
        tracklets_with_emb,
        config=assoc_config,
    )

    print(f'\n  Candidate pairs evaluated: {len(result.candidates):,}')
    print(f'  Accepted matches: {len(result.accepted_matches):,}')
    print(f'  Rejected matches: {len(result.rejected_matches):,}')
    print(f'  Unique global identities: {result.mapping.global_track_id.nunique():,}')
    print(f'  Tracklets merged: {len(tracklets) - result.mapping.global_track_id.nunique():,}')

In [ ]:
if SINGLE_CAMERA_MODE:
    print('Single-camera mode — outputs already saved.')
else:
    # ──────────────── Apply Global IDs & Save ────────────────
    print('Applying global IDs to detections...')
    global_tracks = mtmc_reid.apply_global_ids(detections_with_floor, result.mapping)

    # Save isolated legacy outputs; never overwrite Notebook 02 artifacts.
    output_path = TABLES_DIR / 'global_tracks_legacy.csv'
    global_tracks.to_csv(output_path, index=False)
    print(f'Saved {len(global_tracks):,} rows to {output_path.relative_to(PROJECT_ROOT)}')

    # Save reid_mapping_legacy.csv
    mapping_path = TABLES_DIR / 'reid_mapping_legacy.csv'
    result.mapping.to_csv(mapping_path, index=False)
    print(f'Saved mapping ({len(result.mapping):,} tracklets) to {mapping_path.relative_to(PROJECT_ROOT)}')

    # Save reid_report_legacy.json
    report = {
        'schema_version': 1,
        'created_at_utc': datetime.now(timezone.utc).isoformat(),
        'cameras': int(local_tracks.camera_id.nunique()),
        'status': 'cross_camera_reid',
        'total_tracklets': int(len(tracklets)),
        'global_identities': int(result.mapping.global_track_id.nunique()),
        'accepted_matches': int(len(result.accepted_matches)),
        'rejected_matches': int(len(result.rejected_matches)),
        'candidate_pairs': int(len(result.candidates)),
        'embedding_report': embedding_report,
        'association_config': {
            'max_appearance_distance': assoc_config.max_appearance_distance,
            'max_floor_distance': assoc_config.max_floor_distance,
            'max_time_gap_sec': assoc_config.max_time_gap_sec,
            'max_assignment_score': assoc_config.max_assignment_score,
        },
    }
    report_path = TABLES_DIR / 'reid_report_legacy.json'
    report_path.write_text(json.dumps(report, indent=2), encoding='utf-8')
    print(f'Saved report to {report_path.relative_to(PROJECT_ROOT)}')

    # Print summary
    print(f'\n═══ ReID Summary ═══')
    print(f'  Local tracklets:     {len(tracklets):,}')
    print(f'  Global identities:   {result.mapping.global_track_id.nunique():,}')
    print(f'  Merged (same person): {len(tracklets) - result.mapping.global_track_id.nunique():,}')
    if not result.accepted_matches.empty:
        print(f'\nAccepted cross-camera matches:')
        for _, match in result.accepted_matches.iterrows():
            print(f'  {match["tracklet_id_a"]} ↔ {match["tracklet_id_b"]}  '
                  f'(score={match["score"]:.3f})')

شغّل Notebook 01 أولًا لإنتاج `local_tracks.csv`.

هذا النوتبوك ينتج:
- `global_tracks_legacy.csv` — كل الاكتشافات البحثية مع `global_track_id`
- `reid_mapping_legacy.csv` — جدول ربط Legacy منفصل
- `reid_report_legacy.json` — تقرير Legacy منفصل

بعد هذا النوتبوك، شغّل Notebook 02 ثم 03 لتحديث الـ zones والتحليلات.